## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

sns.set_style('darkgrid')
print("All imports done!")

All imports done!


## Load featured dataset

In [2]:
# ── Load featured_transactions.csv ────────────────────────
# 80 columns: 79 features + Is Laundering
# This is the source of truth for all models

df_feat = pd.read_csv(r"C:\Users\tjain\Downloads\HI-Small_Trans.csv")

print(f"Shape   : {df_feat.shape}")
print(f"Fraud   : {df_feat['Is Laundering'].sum():,}")
print(f"Normal  : {(df_feat['Is Laundering']==0).sum():,}")

y_all = df_feat['Is Laundering'].values

# ── Use encoded account IDs as identifiers ─────────────────
# Your featured CSV has NO original account strings
# Sender_Account_Encoded and Receiver_Account_Encoded
# are the only account identifiers available
# We use them as the join key for all score mappings

SENDER_COL   = 'Sender_Account_Encoded'
RECEIVER_COL = 'Receiver_Account_Encoded'

print(f"\nSender column   : {SENDER_COL}")
print(f"Receiver column : {RECEIVER_COL}")
print(f"Unique senders  : {df_feat[SENDER_COL].nunique():,}")

Shape   : (5078336, 32)
Fraud   : 5,177
Normal  : 5,073,159

Sender column   : Sender_Account_Encoded
Receiver column : Receiver_Account_Encoded
Unique senders  : 496,995


## Load XGBoost model and generate scores for all 5M rows

In [3]:
import joblib
import numpy as np
import os

MODEL_PATH = r"C:\Users\tjain\Downloads\FinShieldAI\models\scaler.pkl"
SCALER_PATH = r"C:\Users\tjain\Downloads\FinShieldAI\models\xgboost.pkl"

model_xgb = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print("XGBoost loaded!")
print(f"Features expected: {model_xgb.n_features_in_}")

XGB_FEATURES = list(model_xgb.feature_names_in_)

print("\nXGBoost feature list:")
for i, f in enumerate(XGB_FEATURES, 1):
    print(f"  {i:2}. {f}")

missing = [f for f in XGB_FEATURES if f not in df_feat.columns]
print(f"\nMissing from featured dataset: {missing}")

df_work = df_feat.copy()

if 'Amount_Received_Log' not in df_work.columns:
    if 'receiver_total_amount_received' in df_work.columns:
        df_work['Amount_Received_Log'] = np.log1p(
            df_work['receiver_total_amount_received'].abs()
        )
    else:
        df_work['Amount_Received_Log'] = 0.0

    print("Created: Amount_Received_Log")

if 'Amount_Diff_Log' not in df_work.columns:
    if (
        'Amount_Paid_Log' in df_work.columns
        and 'Amount_Received_Log' in df_work.columns
    ):
        df_work['Amount_Diff_Log'] = np.log1p(
            np.abs(
                df_work['Amount_Paid_Log']
                - df_work['Amount_Received_Log']
            )
        )
    else:
        df_work['Amount_Diff_Log'] = 0.0

    print("Created: Amount_Diff_Log")

missing = [f for f in XGB_FEATURES if f not in df_work.columns]

if missing:
    print(f"Still missing: {missing}")

    for f in missing:
        df_work[f] = 0.0

    print("Filled with 0.0")

X_xgb = df_work[XGB_FEATURES].fillna(0).astype(np.float32)

print(f"\nX_xgb shape : {X_xgb.shape}")
print(f"NaN count   : {X_xgb.isna().sum().sum()}")
print(f"Inf count   : {np.isinf(X_xgb.values).sum()}")

XGBoost loaded!
Features expected: 29

XGBoost feature list:
   1. Hour
   2. Day
   3. DayOfWeek
   4. IsWeekend
   5. Sender Bank ID
   6. Receiver Bank ID
   7. Sender_Account_Encoded
   8. Receiver_Account_Encoded
   9. is_cross_bank
  10. is_self_transfer
  11. is_currency_mismatch
  12. is_outlier
  13. outlier_score
  14. is_amount_outlier_iqr
  15. is_amount_outlier_zscore
  16. is_account_level_outlier
  17. is_high_velocity
  18. txn_count_per_sender_hour
  19. txn_count_per_sender_day
  20. amount_vs_account_median
  21. Payment_Currency_Encoded
  22. Receiving_Currency_Encoded
  23. fmt_ACH
  24. fmt_Bitcoin
  25. fmt_Cash
  26. fmt_Cheque
  27. fmt_Credit Card
  28. fmt_Reinvestment
  29. fmt_Wire

Missing from featured dataset: []
Created: Amount_Received_Log
Created: Amount_Diff_Log

X_xgb shape : (5078336, 29)
NaN count   : 0
Inf count   : 0


## Score all 5M transactions with XGBoost

In [1]:
xgb_scores = []

batch_size = 100000
n_total = len(X_xgb)

for start in range(0, n_total, batch_size):

    end = min(start + batch_size, n_total)

    batch = X_xgb.iloc[start:end]

    # XGBoost was trained WITHOUT scaler
    probs = model_xgb.predict_proba(batch)[:, 1]

    xgb_scores.extend(probs)

    pct = end / n_total * 100
    print(f"Processed {end:,}/{n_total:,} ({pct:.1f}%)")

xgb_scores = np.array(xgb_scores)

print("\nXGBoost scoring complete!")
print("Number of scores:", len(xgb_scores))
print("Min:", xgb_scores.min())
print("Max:", xgb_scores.max())
print("Mean:", xgb_scores.mean())

NameError: name 'X_xgb' is not defined

In [4]:
# ── Score in batches to avoid memory issues ───────────────
BATCH_SIZE = 200_000
n_total    = len(X_xgb)
xgb_scores = []

print(f"Scoring {n_total:,} transactions in batches of {BATCH_SIZE:,}...")

for start in range(0, n_total, BATCH_SIZE):
    end   = min(start + BATCH_SIZE, n_total)
    batch = X_xgb.iloc[start:end]

    # XGBoost trained WITHOUT scaler in your notebook
    # (your XGBoost notebook used raw featured columns)
    # If it was trained WITH scaler, uncomment next line:
    # batch = scaler.transform(batch)

    probs = model_xgb.predict_proba(batch)[:, 1]
    xgb_scores.extend(probs)

    pct = end / n_total * 100
    if start % 1_000_000 == 0:
        print(f"  {pct:.0f}% done...")

xgb_scores = np.array(xgb_scores, dtype=np.float32)

print(f"\nXGBoost scoring complete!")
print(f"Total scored      : {len(xgb_scores):,}")
print(f"Mean score(fraud) : {xgb_scores[y_all==1].mean():.4f}")
print(f"Mean score(normal): {xgb_scores[y_all==0].mean():.4f}")
print(f"Max score         : {xgb_scores.max():.4f}")

# Quick AUC check
auc_xgb = roc_auc_score(y_all, xgb_scores)
print(f"AUC-ROC (all 5M)  : {auc_xgb:.4f}")

Scoring 5,078,336 transactions in batches of 200,000...


AttributeError: 'RobustScaler' object has no attribute 'predict_proba'

## Score all 5M transactions with XGBoost

In [ ]:
# ── Score in batches to avoid memory issues ───────────────
BATCH_SIZE = 200_000
n_total    = len(X_xgb)
xgb_scores = []

print(f"Scoring {n_total:,} transactions in batches of {BATCH_SIZE:,}...")

for start in range(0, n_total, BATCH_SIZE):
    end   = min(start + BATCH_SIZE, n_total)
    batch = X_xgb.iloc[start:end]

    # XGBoost trained WITHOUT scaler in your notebook
    # (your XGBoost notebook used raw featured columns)
    # If it was trained WITH scaler, uncomment next line:
    # batch = scaler.transform(batch)

    probs = model_xgb.predict_proba(batch)[:, 1]
    xgb_scores.extend(probs)

    pct = end / n_total * 100
    if start % 1_000_000 == 0:
        print(f"  {pct:.0f}% done...")

xgb_scores = np.array(xgb_scores, dtype=np.float32)

print(f"\nXGBoost scoring complete!")
print(f"Total scored      : {len(xgb_scores):,}")
print(f"Mean score(fraud) : {xgb_scores[y_all==1].mean():.4f}")
print(f"Mean score(normal): {xgb_scores[y_all==0].mean():.4f}")
print(f"Max score         : {xgb_scores.max():.4f}")

# Quick AUC check
auc_xgb = roc_auc_score(y_all, xgb_scores)
print(f"AUC-ROC (all 5M)  : {auc_xgb:.4f}")

## Load all other model scores

In [ ]:
# ══════════════════════════════════════════════════════════
# LOAD ALL MODEL SCORES   (corrected)
# ══════════════════════════════════════════════════════════
# FIX 1 — wrong filenames. This cell used to read:
#     results/lstm_scores.csv    (never created — it is lstm_predictions.csv)
#     results/ae_scores.csv      (never created — no autoencoder was run)
#     results/graph_scores.csv   (never created — graph saved graph_node_analysis.csv)
#   All three silently fell into `except` and were reported as
#   "NO SIGNAL", when in reality they were simply never loaded.
#
# FIX 2 — "has signal" used to mean "not all zeros". That is not a
#   signal test. We now measure AUC-ROC against the real labels and
#   record COVERAGE, and let the weighting cell decide from those.
# ══════════════════════════════════════════════════════════

from sklearn.metrics import roc_auc_score

N = len(df_feat)
SCORE_INFO = {}          # name -> dict(coverage, auc, usable)


def register_score(name, values, note=''):
    """Store a score array and measure how good and how complete it is."""
    v = np.nan_to_num(np.asarray(values, dtype=np.float32), nan=0.0)
    coverage = float((v > 0).sum()) / N
    try:
        auc = float(roc_auc_score(y_all, v)) if v.max() > v.min() else 0.5
    except Exception:
        auc = 0.5
    SCORE_INFO[name] = dict(coverage=coverage, auc=auc, note=note)
    print(f"  {name:16} coverage={coverage*100:6.2f}%   AUC-ROC={auc:.4f}   {note}")
    return v


print("Loading all model scores...")
print(f"{'':18}{'coverage':>10}{'':4}{'AUC-ROC':>8}")

# ── XGBoost (already scored earlier in this notebook) ─────
xgb_scores_raw = register_score('xgb_score', xgb_scores, 'full coverage')

# ── LSTM  → results/lstm_predictions.csv ──────────────────
lstm_scores_raw = np.zeros(N, dtype=np.float32)
try:
    lstm_df = pd.read_csv('results/lstm_predictions.csv')
    col = next((c for c in ['lstm_anomaly_score', 'lstm_fraud_score', 'lstm_score']
                if c in lstm_df.columns), None)
    if col is None:
        raise ValueError(f'no score column in {lstm_df.columns.tolist()}')
    if len(lstm_df) == N:
        lstm_scores_raw = lstm_df[col].fillna(0).values.astype(np.float32)
    else:
        # test-set-only file: cannot be aligned to all 5M rows by position
        raise ValueError(
            f'file has {len(lstm_df):,} rows but the dataset has {N:,}. '
            'Re-run the LSTM notebook so it scores every transaction, '
            'or leave LSTM out of the ensemble.')
    lstm_scores_raw = register_score('lstm_score', lstm_scores_raw)
except Exception as e:
    print(f"  lstm_score       NOT USABLE — {e}")
    SCORE_INFO['lstm_score'] = dict(coverage=0.0, auc=0.5, note=str(e)[:80])

# ── CNN → results/cnn_scores.csv (account-level) ──────────
cnn_scores_raw = np.zeros(N, dtype=np.float32)
try:
    cnn_df = pd.read_csv('results/cnn_scores.csv')
    cnn_map = cnn_df.groupby('account_id')['cnn_fraud_score'].mean()
    cnn_scores_raw = (df_feat[SENDER_COL].map(cnn_map)
                      .fillna(0).values.astype(np.float32))
    cnn_scores_raw = register_score(
        'cnn_score', cnn_scores_raw,
        'account-level — only accounts the CNN was trained on')
except Exception as e:
    print(f"  cnn_score        NOT USABLE — {e}")
    SCORE_INFO['cnn_score'] = dict(coverage=0.0, auc=0.5, note=str(e)[:80])

# ── Graph → featured/featured_with_graph.csv ──────────────
graph_scores_raw = np.zeros(N, dtype=np.float32)
try:
    gcols = ['graph_risk_score', 'graph_mule_score']
    gdf = pd.read_csv('featured/featured_with_graph.csv', usecols=lambda c: c in gcols)
    gcol = next((c for c in gcols if c in gdf.columns), None)
    if gcol is None or len(gdf) != N:
        raise ValueError('graph risk columns not found at full length')
    g = gdf[gcol].fillna(0).values.astype(np.float32)
    graph_scores_raw = register_score('graph_score', g / max(g.max(), 1e-9),
                                      f'from {gcol}')
except Exception as e:
    print(f"  graph_score      NOT USABLE — {e}")
    SCORE_INFO['graph_score'] = dict(coverage=0.0, auc=0.5, note=str(e)[:80])

# ── Autoencoder — genuinely not built in this project ─────
ae_scores_raw = np.zeros(N, dtype=np.float32)
SCORE_INFO['ae_score'] = dict(coverage=0.0, auc=0.5,
                              note='no autoencoder notebook in this project')
print("  ae_score         NOT USABLE — no autoencoder was trained")

# ── Composite rule score ──────────────────────────────────
composite_raw = np.zeros(N, dtype=np.float32)
try:
    ccol = next((c for c in ['composite_risk_score', 'outlier_score']
                 if c in df_feat.columns), None)
    c = df_feat[ccol].fillna(0).values.astype(np.float32)
    composite_raw = register_score('composite_score', c / max(c.max(), 1e-9),
                                   f'from {ccol}')
except Exception as e:
    print(f"  composite_score  NOT USABLE — {e}")
    SCORE_INFO['composite_score'] = dict(coverage=0.0, auc=0.5, note=str(e)[:80])

print("\nAUC-ROC of 0.50 means the score is no better than random.")


## Build master risk DataFrame

In [ ]:
# ── Build clean master DataFrame ──────────────────────────
risk_df = pd.DataFrame({
    'Is Laundering'          : y_all,
    'Sender_Account_Encoded' : df_feat[SENDER_COL].values,
    'xgb_score'              : xgb_scores,
    'lstm_score'             : lstm_scores_raw,
    'ae_score'               : ae_scores_raw,
    'cnn_score'              : cnn_scores_raw,
    'graph_score'            : graph_scores_raw,
    'composite_score'        : composite_raw,
})

# Add Amount Paid for display
if 'Amount_Paid_Log' in df_feat.columns:
    risk_df['Amount_Paid_Log'] = df_feat['Amount_Paid_Log'].values

print(f"Master risk DataFrame: {risk_df.shape}")
print(f"\nScore ranges:")
for col in ['xgb_score','lstm_score','ae_score','cnn_score','graph_score','composite_score']:
    vals = risk_df[col].values
    print(f"  {col:20}: min={vals.min():.4f} max={vals.max():.4f} mean={vals.mean():.4f}")

## Weighted ensemble scoring

In [ ]:
# ══════════════════════════════════════════════════════════
# WEIGHTED ENSEMBLE   (corrected)
# ══════════════════════════════════════════════════════════
# WHAT WAS WRONG BEFORE
#   1. BASE_WEIGHTS hard-coded a weight for every model BEFORE any of
#      them were evaluated, then just renormalised among whichever
#      files happened to load. Nothing was weighted by performance.
#   2. "has signal" only checked (score > 0).sum() > 1000 — it never
#      checked whether the score PREDICTED anything. CNN passed that
#      test with an AUC-ROC of 0.5172 (random) and kept 14.7% weight.
#   3. A model covering 3% of rows was blended as if it covered 100%;
#      the other 97% silently received a hard 0.
#   4. Nothing compared the finished ensemble against its own best
#      single model. The saved config recorded ensemble AUC 0.7039 vs
#      XGBoost 0.9110 — far worse — and shipped anyway.
#
# NEW RULE: a model earns weight only if it clears BOTH gates —
#   MIN_AUC        it must actually separate fraud from normal
#   MIN_COVERAGE   it must score most of the dataset
# Surviving models are then weighted by how far above random they are
# (auc - 0.5), so a better model automatically gets more say.
# ══════════════════════════════════════════════════════════

MIN_AUC      = 0.55   # below this a score is ~random and is excluded
MIN_COVERAGE = 0.80   # must cover at least 80% of transactions

CANDIDATES = ['xgb_score', 'lstm_score', 'ae_score',
              'cnn_score', 'graph_score', 'composite_score']

print("Model admission (must pass BOTH gates):")
print(f"  gates: AUC-ROC >= {MIN_AUC}   coverage >= {MIN_COVERAGE*100:.0f}%\n")

admitted = {}
for col in CANDIDATES:
    info = SCORE_INFO.get(col, dict(coverage=0.0, auc=0.5, note='not loaded'))
    auc, cov = info['auc'], info['coverage']
    ok_auc, ok_cov = auc >= MIN_AUC, cov >= MIN_COVERAGE
    if ok_auc and ok_cov:
        admitted[col] = auc - 0.5      # skill above random
        verdict = 'ADMITTED'
    else:
        reasons = []
        if not ok_auc: reasons.append(f'AUC {auc:.4f} < {MIN_AUC}')
        if not ok_cov: reasons.append(f'coverage {cov*100:.1f}% < {MIN_COVERAGE*100:.0f}%')
        verdict = 'EXCLUDED — ' + '; '.join(reasons)
    print(f"  {col:16} AUC={auc:.4f}  cov={cov*100:6.2f}%   {verdict}")

if not admitted:
    raise RuntimeError("No model passed the gates — cannot build an ensemble.")

total_skill = sum(admitted.values())
WEIGHTS = {col: 0.0 for col in CANDIDATES}
for col, skill in admitted.items():
    WEIGHTS[col] = skill / total_skill

print("\nFinal weights (proportional to skill above random):")
for col, w in WEIGHTS.items():
    if w > 0:
        print(f"  {col:16}: {w:.4f} ({w*100:.1f}%)")
print(f"  sum = {sum(WEIGHTS.values()):.6f}")

# ── Compute the ensemble ──────────────────────────────────
risk_df['ensemble_raw'] = sum(WEIGHTS[c] * risk_df[c] for c in WEIGHTS)

# ══════════════════════════════════════════════════════════
# SAFETY GUARD — never ship a blend worse than its best member
# ══════════════════════════════════════════════════════════
from sklearn.metrics import roc_auc_score

auc_ensemble = roc_auc_score(y_all, risk_df['ensemble_raw'])
best_single_col = max(admitted, key=lambda c: SCORE_INFO[c]['auc'])
auc_best_single = SCORE_INFO[best_single_col]['auc']

print(f"\nEnsemble AUC-ROC     : {auc_ensemble:.4f}")
print(f"Best single ({best_single_col}) : {auc_best_single:.4f}")

if auc_ensemble < auc_best_single:
    print("\n  GUARD TRIGGERED — the blend is WORSE than its best single model.")
    print(f"  Falling back to {best_single_col} alone.")
    WEIGHTS = {c: (1.0 if c == best_single_col else 0.0) for c in CANDIDATES}
    risk_df['ensemble_raw'] = risk_df[best_single_col]
    auc_ensemble = auc_best_single
else:
    print("  Guard passed — the blend beats every individual model.")

# ── Scale to 0-100 by PERCENTILE RANK ─────────────────────
# FIX: min-max scaling was previously used, which lets one extreme
# value squash everything else into a narrow band and makes the
# 0-100 number impossible to interpret. Percentile rank means
# "this transaction is riskier than X% of all transactions" — and it
# is what the top-0.5% / 2.5% / 10% tiers below actually assume.
risk_df['risk_score_100'] = risk_df['ensemble_raw'].rank(pct=True) * 100

print(f"\nEnsemble score stats:")
print(f"  Min  : {risk_df['risk_score_100'].min():.4f}")
print(f"  Max  : {risk_df['risk_score_100'].max():.4f}")
print(f"  Mean : {risk_df['risk_score_100'].mean():.4f}")
print(f"  Fraud mean  : {risk_df.loc[y_all==1,'risk_score_100'].mean():.4f}")
print(f"  Normal mean : {risk_df.loc[y_all==0,'risk_score_100'].mean():.4f}")


## Calibrate thresholds from actual score distribution

In [ ]:
# ════════════════════════════════════════════════════════
# THRESHOLD CALIBRATION
# ════════════════════════════════════════════════════════
# We do NOT use fixed 30/50/70 thresholds
# We calibrate from actual score distribution so that:
#   CRITICAL = top 0.5% of all transactions
#   HIGH     = top 2.5% of all transactions
#   MEDIUM   = top 10% of all transactions
#   LOW      = everything else
#
# This gives manageable alert volumes regardless of
# what the raw score distribution looks like
# ════════════════════════════════════════════════════════

scores = risk_df['risk_score_100'].values

# Percentile-based thresholds
THRESH_CRITICAL = np.percentile(scores, 99.5)   # top 0.5%
THRESH_HIGH     = np.percentile(scores, 97.5)   # top 2.5%
THRESH_MEDIUM   = np.percentile(scores, 90.0)   # top 10%

print(f"Calibrated thresholds from actual score distribution:")
print(f"  CRITICAL (top 0.5%) : score >= {THRESH_CRITICAL:.2f}")
print(f"  HIGH     (top 2.5%) : score >= {THRESH_HIGH:.2f}")
print(f"  MEDIUM   (top 10%)  : score >= {THRESH_MEDIUM:.2f}")
print(f"  LOW                 : everything else")

def classify_risk(score):
    if score >= THRESH_CRITICAL: return 'CRITICAL'
    elif score >= THRESH_HIGH:   return 'HIGH'
    elif score >= THRESH_MEDIUM: return 'MEDIUM'
    else:                        return 'LOW'

risk_df['risk_level'] = risk_df['risk_score_100'].apply(classify_risk)

print(f"\nRisk level distribution:")
for lvl in ['LOW','MEDIUM','HIGH','CRITICAL']:
    sub  = risk_df[risk_df['risk_level']==lvl]
    rate = sub['Is Laundering'].mean() * 100
    n    = len(sub)
    bar  = '█' * int(rate * 8)
    print(f"  {lvl:8}: {n:>10,} txns | fraud rate: {rate:.4f}%  {bar}")

print(f"\nAlerts (HIGH + CRITICAL): {(risk_df['risk_level'].isin(['HIGH','CRITICAL'])).sum():,}")
print(f"Alert rate              : {(risk_df['risk_level'].isin(['HIGH','CRITICAL'])).mean()*100:.2f}%")

## Fraud capture analysis

In [ ]:
# ── Fraud capture by risk level ───────────────────────────
total_fraud = int(y_all.sum())
print(f"Total fraud in dataset: {total_fraud:,}")

print(f"\nFraud captured by risk level:")
cumulative = 0
for lvl in ['CRITICAL','HIGH','MEDIUM','LOW']:
    sub       = risk_df[(risk_df['risk_level']==lvl) & (risk_df['Is Laundering']==1)]
    n_fraud   = len(sub)
    cumulative += n_fraud
    pct       = n_fraud / total_fraud * 100
    print(f"  {lvl:8}: {n_fraud:>5,} fraud ({pct:.1f}%) | cumulative: {cumulative:,} ({cumulative/total_fraud*100:.1f}%)")

# ── Generate alerts ───────────────────────────────────────
alerts = risk_df[
    risk_df['risk_level'].isin(['CRITICAL','HIGH'])
].copy().sort_values('risk_score_100', ascending=False)

alerts['alert_id']     = [f"AML-{i+1:06d}" for i in range(len(alerts))]
alerts['alert_status'] = 'PENDING_REVIEW'

fraud_in_alerts = int(alerts['Is Laundering'].sum())
capture_rate    = fraud_in_alerts / total_fraud * 100
alert_rate      = len(alerts) / len(risk_df) * 100

print(f"\n{'='*55}")
print(f"ALERT SUMMARY")
print(f"{'='*55}")
print(f"  Total alerts     : {len(alerts):,}")
print(f"  Alert rate       : {alert_rate:.2f}%")
print(f"  Fraud captured   : {fraud_in_alerts:,} / {total_fraud:,}")
print(f"  Capture rate     : {capture_rate:.2f}%")

print(f"\nTop 10 highest risk alerts:")
display_cols = ['alert_id','risk_level','risk_score_100','xgb_score','Is Laundering']
if 'Amount_Paid_Log' in alerts.columns:
    display_cols.insert(4, 'Amount_Paid_Log')
print(alerts[display_cols].head(10).to_string(index=False))

## Comprehensive visualization

In [ ]:
fig = plt.figure(figsize=(24, 18))
fig.suptitle(
    f'FinShield AI — Risk Scoring Engine\n'
    f'XGBoost Ensemble | {len(alerts):,} Alerts | '
    f'{fraud_in_alerts}/{total_fraud} Fraud Captured ({capture_rate:.1f}%)',
    fontsize=14, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

colors_risk = {
    'LOW':'#4CAF50', 'MEDIUM':'#FF9800',
    'HIGH':'#E53935', 'CRITICAL':'#7B1FA2'
}
level_order = ['LOW','MEDIUM','HIGH','CRITICAL']

# Chart 1: Risk level distribution
ax1 = fig.add_subplot(gs[0,0])
cnts  = risk_df['risk_level'].value_counts()
bars1 = ax1.bar(
    [l for l in level_order if l in cnts],
    [cnts.get(l,0) for l in level_order if l in cnts],
    color=[colors_risk[l] for l in level_order if l in cnts],
    edgecolor='white', width=0.6
)
ax1.set_title('Transaction Risk Distribution', fontweight='bold')
ax1.set_ylabel('Count')
for bar in bars1:
    ax1.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+5000,
        f'{int(bar.get_height()):,}',
        ha='center', fontsize=8, fontweight='bold'
    )

# Chart 2: Fraud rate by risk level
ax2 = fig.add_subplot(gs[0,1])
fraud_rates = [
    risk_df[risk_df['risk_level']==l]['Is Laundering'].mean()*100
    for l in level_order
]
bars2 = ax2.bar(
    level_order, fraud_rates,
    color=[colors_risk[l] for l in level_order],
    edgecolor='white', width=0.6
)
ax2.set_title('Fraud Rate by Risk Level', fontweight='bold')
ax2.set_ylabel('Fraud Rate (%)')
for bar, rate in zip(bars2, fraud_rates):
    ax2.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+0.001,
        f'{rate:.4f}%', ha='center', fontweight='bold', fontsize=9
    )

# Chart 3: Risk score distribution (fraud vs normal)
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(
    risk_df[risk_df['Is Laundering']==0]['risk_score_100'],
    bins=60, color='#2196F3', alpha=0.6, label='Normal', density=True
)
ax3.hist(
    risk_df[risk_df['Is Laundering']==1]['risk_score_100'],
    bins=60, color='#E53935', alpha=0.6, label='Fraud',  density=True
)
ax3.axvline(THRESH_HIGH,     color='orange', linestyle='--', linewidth=2, label=f'HIGH ({THRESH_HIGH:.1f})')
ax3.axvline(THRESH_CRITICAL, color='purple', linestyle='--', linewidth=2, label=f'CRITICAL ({THRESH_CRITICAL:.1f})')
ax3.set_title('Risk Score Distribution', fontweight='bold')
ax3.set_xlabel('Risk Score (0-100)')
ax3.legend(fontsize=8)

# Chart 4: XGBoost score distribution
ax4 = fig.add_subplot(gs[1,0])
ax4.hist(
    risk_df[risk_df['Is Laundering']==0]['xgb_score'],
    bins=60, color='#2196F3', alpha=0.6, label='Normal', density=True
)
ax4.hist(
    risk_df[risk_df['Is Laundering']==1]['xgb_score'],
    bins=60, color='#E53935', alpha=0.6, label='Fraud',  density=True
)
ax4.set_title('XGBoost Score Distribution', fontweight='bold')
ax4.set_xlabel('XGBoost Fraud Probability')
ax4.legend(fontsize=8)

# Chart 5: Model weights pie
ax5 = fig.add_subplot(gs[1,1])
active_w = {k: v for k, v in WEIGHTS.items() if v > 0}
ax5.pie(
    list(active_w.values()),
    labels=[f"{k.replace('_score','').replace('_',' ').title()}\n({v*100:.0f}%)"
            for k,v in active_w.items()],
    colors=['#E53935','#FF9800','#2196F3','#9C27B0','#4CAF50','#795548'][:len(active_w)],
    autopct='%1.0f%%', startangle=90, pctdistance=0.75
)
ax5.set_title('Ensemble Weights', fontweight='bold')

# Chart 6: Alert volume
ax6 = fig.add_subplot(gs[1,2])
alert_cnts = {l: len(risk_df[risk_df['risk_level']==l]) for l in level_order}
ax6.barh(
    level_order,
    [alert_cnts[l] for l in level_order],
    color=[colors_risk[l] for l in level_order],
    edgecolor='white'
)
ax6.set_title('Transaction Count by Level', fontweight='bold')
ax6.set_xlabel('Count')
for i, (l, cnt) in enumerate(alert_cnts.items()):
    ax6.text(cnt + 500, i, f'{cnt:,}', va='center', fontsize=8)

# Chart 7: Fraud capture curve
ax7 = fig.add_subplot(gs[2,0])
pcts          = np.arange(0, 101, 1)
thresholds_pct= np.percentile(risk_df['risk_score_100'], pcts)
capture_curve = []
alert_curve   = []
for t in thresholds_pct[::-1]:
    flagged     = risk_df[risk_df['risk_score_100'] >= t]
    cap         = flagged['Is Laundering'].sum() / max(total_fraud, 1) * 100
    alt         = len(flagged) / len(risk_df) * 100
    capture_curve.append(cap)
    alert_curve.append(alt)

ax7.plot(alert_curve, capture_curve, color='#E53935', linewidth=2.5,
         label='Fraud captured vs Alert volume')
ax7.plot([0,100],[0,100], '--', color='gray', linewidth=1, label='Random')
ax7.set_title('Fraud Capture Curve\n(Lift over Random)', fontweight='bold')
ax7.set_xlabel('Alert Volume (%)')
ax7.set_ylabel('Fraud Captured (%)')
ax7.legend(fontsize=8)

# Chart 8: XGBoost vs Ensemble scatter
ax8 = fig.add_subplot(gs[2,1])
n_samp    = min(5000, len(risk_df))
samp_idx  = np.random.choice(len(risk_df), n_samp, replace=False)
samp_data = risk_df.iloc[samp_idx]
sc = ax8.scatter(
    samp_data['xgb_score'],
    samp_data['risk_score_100'],
    c=samp_data['Is Laundering'],
    cmap='RdBu_r', alpha=0.4, s=8,
    vmin=0, vmax=1
)
ax8.set_title('XGBoost vs Ensemble Score\nRed=Fraud, Blue=Normal', fontweight='bold')
ax8.set_xlabel('XGBoost Score')
ax8.set_ylabel('Ensemble Risk Score (0-100)')
plt.colorbar(sc, ax=ax8)

# Chart 9: ROC curve
ax9 = fig.add_subplot(gs[2,2])
fpr_x, tpr_x, _ = roc_curve(y_all, risk_df['xgb_score'])
fpr_e, tpr_e, _ = roc_curve(y_all, risk_df['risk_score_100'])
auc_x = roc_auc_score(y_all, risk_df['xgb_score'])
auc_e = roc_auc_score(y_all, risk_df['risk_score_100'])
ax9.plot(fpr_x, tpr_x, color='#E53935', linewidth=2,
         label=f'XGBoost (AUC={auc_x:.4f})')
ax9.plot(fpr_e, tpr_e, color='black', linewidth=2.5, linestyle='--',
         label=f'Ensemble (AUC={auc_e:.4f})')
ax9.plot([0,1],[0,1],'--', color='lightgray', linewidth=1, label='Random')
ax9.set_title('ROC Curve', fontweight='bold')
ax9.set_xlabel('FPR'); ax9.set_ylabel('TPR')
ax9.legend(fontsize=8)

plt.savefig('risk_engine_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved!")

## Save everything

In [ ]:
os.makedirs('results', exist_ok=True)
os.makedirs('models',  exist_ok=True)

# Save full risk scores
risk_df.to_csv('results/final_risk_scores.csv', index=False)
print(f"Saved: results/final_risk_scores.csv ({len(risk_df):,} rows)")

# Save alerts
alerts.to_csv('results/alerts.csv', index=False)
print(f"Saved: results/alerts.csv ({len(alerts):,} alerts)")

# Save risk engine config (Flask API loads this)
risk_config = {
    'weights'            : WEIGHTS,
    'thresholds'         : {
        'CRITICAL': float(THRESH_CRITICAL),
        'HIGH'    : float(THRESH_HIGH),
        'MEDIUM'  : float(THRESH_MEDIUM),
        'LOW'     : 0.0
    },
    'total_transactions' : int(len(risk_df)),
    'total_alerts'       : int(len(alerts)),
    'fraud_captured'     : int(fraud_in_alerts),
    'total_fraud'        : int(total_fraud),
    'capture_rate_pct'   : round(capture_rate, 2),
    'alert_rate_pct'     : round(alert_rate, 2),
    'active_models'      : [col for col, w in WEIGHTS.items() if w > 0],
    'xgb_features'       : XGB_FEATURES,
    'auc_xgb'            : round(auc_x, 4),
    'auc_ensemble'       : round(auc_e, 4),
    # ── added by the corrected weighting cell ──
    'score_diagnostics'  : {k: {'auc': round(v['auc'], 4),
                                'coverage': round(v['coverage'], 4)}
                            for k, v in SCORE_INFO.items()},
    'admission_gates'    : {'min_auc': MIN_AUC, 'min_coverage': MIN_COVERAGE},
    'scaling'            : 'percentile_rank',
}
joblib.dump(risk_config, 'models/risk_engine_config.pkl')
print(f"Saved: models/risk_engine_config.pkl")

## Final summary

In [ ]:
print("=" * 60)
print("RISK SCORING ENGINE — COMPLETE")
print("=" * 60)

print(f"""
ACTIVE MODELS AND WEIGHTS
{chr(10).join(f"  {col.replace('_score',''):15}: {w:.3f} ({w*100:.1f}%)"
              for col, w in WEIGHTS.items() if w > 0)}

PERFORMANCE
  XGBoost AUC-ROC   : {auc_x:.4f}
  Ensemble AUC-ROC  : {auc_e:.4f}

CALIBRATED THRESHOLDS (from actual score distribution)
  CRITICAL >= {THRESH_CRITICAL:.2f}  (top 0.5% of all transactions)
  HIGH     >= {THRESH_HIGH:.2f}  (top 2.5%)
  MEDIUM   >= {THRESH_MEDIUM:.2f}  (top 10%)
  LOW         everything else

RISK DISTRIBUTION
  CRITICAL : {(risk_df['risk_level']=='CRITICAL').sum():>10,} ({(risk_df['risk_level']=='CRITICAL').mean()*100:.2f}%)
  HIGH     : {(risk_df['risk_level']=='HIGH').sum():>10,} ({(risk_df['risk_level']=='HIGH').mean()*100:.2f}%)
  MEDIUM   : {(risk_df['risk_level']=='MEDIUM').sum():>10,} ({(risk_df['risk_level']=='MEDIUM').mean()*100:.2f}%)
  LOW      : {(risk_df['risk_level']=='LOW').sum():>10,} ({(risk_df['risk_level']=='LOW').mean()*100:.2f}%)

ALERTS
  Total generated    : {len(alerts):,} ({alert_rate:.2f}% of all transactions)
  Fraud captured     : {fraud_in_alerts:,} / {total_fraud:,}
  Capture rate       : {capture_rate:.2f}%

FILES SAVED
  results/final_risk_scores.csv  ({len(risk_df):,} rows)
  results/alerts.csv             ({len(alerts):,} alerts)
  models/risk_engine_config.pkl

NEXT → Flask API + Dashboard
""")